In [ ]:
!pip install -q crewai
!pip install -q openai
!pip install -q unstructured
!pip install -q tools
!pip install -q tenacity==8.3.0
!pip install -q langchain
!pip install -q langchain_groq
!pip install -q cohere
!pip install -q langchain_community
!pip install -q 'crewai[tools]'

In [ ]:
from crewai import Agent, Task, Crew
from langchain_community.chat_models import ChatCohere
from langchain_openai import OpenAI
from langchain_groq import ChatGroq


In [ ]:
#warning control
import warnings
warnings.filterwarnings('ignore')

In [ ]:
import os

COHERE_API_KEY="ieLPQ8jIDR7czKpyAcRYPJTdjjP27AAVwR7Gvwzs"
OPENAI_API_KEY="sk-crew-ai-gqQfhuNQaIn8AKoIRg8UT3BlbkFJfEam04Z3SQEPwjWabC4Y"
GROQ_API_KEY="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"
SERPER_API_KEY= "25b084c4ee79d46c1866395746dbdf145c1729d9"


os.environ['OPENAI_API_KEY'] = OPENAI_API_KEY
os.environ['COHERE_API_KEY'] = COHERE_API_KEY
os.environ['GROQ_API_KEY'] = GROQ_API_KEY
os.environ['SERPER_API_KEY'] = SERPER_API_KEY

#Tools
from crewai_tools import (
    SerperDevTool,
    WebsiteSearchTool
)

search_tool = SerperDevTool()
web_search_tool = WebsiteSearchTool()

#LLMs

cohere = ChatCohere(cohere_api_key=COHERE_API_KEY,
                    temperaature= 0.3)
openai = OpenAI(api_key=OPENAI_API_KEY)
groq = ChatGroq(
                temperature=0,
                groq_api_key=GROQ_API_KEY,
                model_name="mixtral-8x7b-32768"
            )


In [ ]:
#Agent
pain_points_analyst = Agent(
                         role='Business Pain Points Analyst',
                        goal="""Identify pain points of Job Titles within each company provided by the {report},
                                 ranking them by their contribution to loss of revenue.""",
                        backstory="""You are a Business Pain Points Analyst who identifies the key pain points of job titles provided by the {report},
                                      You Rank the key pain points in order of the intensity of their impact on revenue. As part of your final submission you always submit a three column table.
                                      The heading for the first column is named Job title, and the cells column are  all the Job titles you are identified. The heading for the second
                                      column  is named Ranked pain points and  cells in this column are the pain points you identifed. The heading for the third column is named
                                      Impact On Revenue, and the cells in this column has the ranking for the respective  job title and ranked pain points.
                                      You also filter the pain points and identify the solutions that were used to address them in the past. You base your ranking on reviews and sentiments from the targeted Job Titles.""",
                        # tools=[
                        #         search_tool
                        #       ],
                        allow_delegation=False,
                        verbose=True,
                        llm=groq
                       )

In [ ]:
painPoints = Task (
       description= """
                       Identify pain points of Job Titles provided {report}. These pain points should be ranked in order of the intensity of the impact on revenue.
                       Filter the pain points and identify the solutions that were used to address them. You based your ranking on reviews and sentiments from the targeted Job Titles.
                       Expand this analysis into a comprehensive report with a detailed per-step plan, including identification, ranking, filtering, and solution identification.
                       You MUST identify specific pain points, rank them by impact on revenue, filter them to highlight the most critical ones, and suggest actual solutions used to address these pain points.
                       This report should cover all aspects of the analysis, from initial identification to final solutions, integrating insights from the Business Portfolio Analyst with practical business impacts.
                       Your final answer MUST be a complete expanded report, formatted as markdown, encompassing each step of the process, detailed explanations of pain points, their rankings,
                       identified solutions, and reasons for selecting each solution, ensuring the most thorough understanding of the issues and resolutions.
                       Be specific and explain why each pain point and solution was chosen, and what makes them significant!
                   """,
            expected_output="A comprehensive detail of the pain points of Job Titles provided {report}.",
            agent=pain_points_analyst)

In [ ]:
#Crew
crew = Crew(
    agents=[pain_points_analyst],
    tasks=[painPoints],
    verbose=True,
)

In [ ]:
#Execute Crew
result = crew.kickoff(inputs={
    "report": """
              ## **1. Tesla, Inc. (Subniche: Electric Vehicles and Sustainable Transport)**

              ### **Job Titles:**
              - **Chief Technology Officer (CTO):** Responsible for overseeing technology development, including battery systems and autonomous driving.
              - **Vice President of Production:** In charge of managing production operations and addressing capacity challenges.
              - **Head of Regulatory Affairs:** Leads the team navigating legal and safety standards for autonomous vehicles.

             ### **Supervisors:**
             - **CTO's Supervisor:** The CTO reports directly to the Chief Executive Officer (CEO) of Tesla, who is responsible for the overall technology strategy and innovation.
             - **Vice President of Production's Supervisor:** This role typically reports to the Chief Operating Officer (COO) or a senior executive overseeing manufacturing and operations.
             - **Head of Regulatory Affairs' Supervisor:** This position is supervised by the General Counsel or a senior legal officer, ensuring compliance with regulations.

             ## **2. Toyota Motor Corporation (Subniche: Hybrid Vehicles and Lean Manufacturing)**

             ### **Job Titles:**
             - **Chief Engineer (Hybrid Systems):** Leads the development and improvement of hybrid vehicle technology.
             - **Supply Chain Director:** Responsible for managing the supply chain, ensuring sustainable sourcing and efficient logistics.
             - **Global Market Strategy Manager:** Develops and implements strategies for adapting vehicles to different global markets.

             ### **Supervisors:**
             - **Chief Engineer's Supervisor:** The Chief Engineer reports to the Executive Vice President of Engineering, who oversees all engineering functions within Toyota.
             - **Supply Chain Director's Supervisor:** This role typically reports to the Chief Procurement Officer or a senior executive in charge of supply chain management.
             - **Global Market Strategy Manager's Supervisor:** This position is supervised by the Chief Marketing Officer or a senior marketing executive, ensuring alignment with global branding and sales strategies.

          ## **3. Mercedes-Benz (Subniche: Luxury Automobiles and Advanced Driver Assistance)**

           ### **Job Titles:**
             - **Chief Customer Officer (CCO):** Focuses on enhancing the customer experience and personalization.
             - **Head of Autonomous Driving:** Leads the development and implementation of advanced driver assistance systems.
             - **Digital Transformation Officer:** Responsible for driving digital initiatives and technology integration.

           ### **Supervisors:**
              - **CCO's Supervisor:** The CCO reports to the CEO or a senior executive responsible for overall customer satisfaction and brand experience.
              - **Head of Autonomous Driving's Supervisor:** This role is supervised by the Chief Technology Officer or a senior engineering executive, ensuring alignment with safety and regulatory requirements.
              - **Digital Transformation Officer's Supervisor:** Typically reports to the Chief Information Officer (CIO) or a senior executive overseeing digital strategy and innovation.

           ## **4. Harley-Davidson (Subniche: Motorcycles and Lifestyle Brand)**

             ### **Job Titles:**
               - **Chief Brand Officer (CBO):** Oversees brand strategy, including the transition to electric motorcycles while preserving brand identity.
               - **Community Engagement Manager:** Responsible for creating and managing digital communities and experiences.
               - **International Business Development Director:** Leads the expansion into new global markets.

             ### **Supervisors:**
              - **CBO's Supervisor:** The CBO reports to the CEO or a senior executive responsible for brand management and marketing.
              - **Community Engagement Manager's Supervisor:** This role typically reports to the CBO or a senior marketing executive, ensuring alignment with brand engagement strategies.
              - **International Business Development Director's Supervisor:** Reports to the Chief Executive Officer or a senior executive overseeing global business development and expansion.

             ## **5. Uber Technologies, Inc. (Subniche: Ride-Hailing and Mobility Services)**

             ### **Job Titles:**
             - **Chief Legal Officer (CLO):** Navigates legal and regulatory challenges across various markets.
             - **Head of Safety:** Develops and implements safety measures for drivers and riders.
             - **Chief Logistics Officer (CLO):** Optimizes logistics operations and fleet management.
             - **Product Manager (Customer Experience):** Enhances the app and customer loyalty programs.

             ### **Supervisors:**
            - **CLO's Supervisor:** The CLO reports to the CEO or a senior executive responsible for overall corporate governance and compliance.
            - **Head of Safety's Supervisor:** This role is supervised by the CLO or a senior executive overseeing legal and safety matters.
              """
     })

 [2024-08-30 13:55:55][DEBUG]: == Working Agent: Business Pain Points Analyst
 [2024-08-30 13:55:55][INFO]: == Starting Task:  
                       Identify pain points of Job Titles provided 
              ## **1. Tesla, Inc. (Subniche: Electric Vehicles and Sustainable Transport)**

### **Job Titles:**
- **Chief Technology Officer (CTO):** Responsible for overseeing technology development, including battery systems and autonomous driving.
- **Vice President of Production:** In charge of managing production operations and addressing capacity challenges.
- **Head of Regulatory Affairs:** Leads the team navigating legal and safety standards for autonomous vehicles.

### **Supervisors:**
- **CTO's Supervisor:** The CTO reports directly to the Chief Executive Officer (CEO) of Tesla, who is responsible for the overall technology strategy and innovation.
- **Vice President of Production's Supervisor:** This role typically reports to the Chief Operating Officer (COO) or a senior executive